In [1]:
from pathlib import Path
import pandas as pd
import re

# ========= 改成你的路徑 =========
input_csv = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/housing_tables/ethnicity_housing_shares.csv")
output_tex = Path(r"/Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/housing_tables/ethnicity_housing_shares.tex")
# ===============================

def latex_escape(text):
    if pd.isna(text):
        return ""
    text = str(text)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group(0)], text)

df = pd.read_csv(input_csv)

cols = [
    "ethn_group",
    "p_bed_lt1",
    "p_bed_1to2",
    "p_bed_ge2",
    "p_room_01",
    "p_room_23",
    "p_room_4p",
    "n",
]
missing = [c for c in cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV 缺少欄位: {missing}")

df = df[cols].copy()

num_cols = [
    "p_bed_lt1",
    "p_bed_1to2",
    "p_bed_ge2",
    "p_room_01",
    "p_room_23",
    "p_room_4p",
    "n",
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

ethnicity_order = [
    "British/English/Scottish/Welsh/Northern Irish",
    "Indian",
    "Pakistani",
    "Bangladeshi",
    "African",
    "Caribbean",
    "Any other white background",
]
df["ethn_group"] = pd.Categorical(
    df["ethn_group"],
    categories=ethnicity_order,
    ordered=True
)
df = df.sort_values("ethn_group").reset_index(drop=True)

df["ethn_group"] = df["ethn_group"].astype(str).map(latex_escape)

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\caption{Housing Conditions at Baseline by Ethnicity}")
lines.append(r"\label{tab:ethnicity_housing}")
lines.append(r"\begin{threeparttable}")
lines.append(r"\footnotesize")
lines.append(r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.4cm}ccccccc}")
lines.append(r"\toprule")
lines.append(r"& \multicolumn{3}{c}{Bedroom-per-person ratio} & \multicolumn{3}{c}{Number of other rooms} & \\")
lines.append(r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}")
lines.append(r"Ethnicity & <1 & 1--<2 & $\geq$2 & 0--1 & 2--3 & 4+ & N \\")
lines.append(r"\midrule")

for _, row in df.iterrows():
    eth = row["ethn_group"]
    b1 = "" if pd.isna(row["p_bed_lt1"]) else f'{row["p_bed_lt1"]:.2f}'
    b2 = "" if pd.isna(row["p_bed_1to2"]) else f'{row["p_bed_1to2"]:.2f}'
    b3 = "" if pd.isna(row["p_bed_ge2"]) else f'{row["p_bed_ge2"]:.2f}'
    r1 = "" if pd.isna(row["p_room_01"]) else f'{row["p_room_01"]:.2f}'
    r2 = "" if pd.isna(row["p_room_23"]) else f'{row["p_room_23"]:.2f}'
    r3 = "" if pd.isna(row["p_room_4p"]) else f'{row["p_room_4p"]:.2f}'
    n = "" if pd.isna(row["n"]) else f'{int(round(row["n"])):,}'
    lines.append(f"{eth} & {b1} & {b2} & {b3} & {r1} & {r2} & {r3} & {n} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular*}")
lines.append(r"\begin{tablenotes}[flushleft]")
lines.append(r"\footnotesize")
lines.append(
    r"\item Notes: This table reports weighted percentages of baseline housing conditions by ethnicity. The first three columns show the distribution of the bedroom-per-person ratio, and the next three columns show the distribution of the number of other rooms at baseline. Percentages are weighted using the CA Covid survey weights, and \(N\) denotes the unweighted sample size."
)
lines.append(r"\end{tablenotes}")
lines.append(r"\end{threeparttable}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
output_tex.write_text(latex_table, encoding="utf-8")

print(f"LaTeX table saved to: {output_tex}")
print()
print(latex_table)

LaTeX table saved to: /Users/lishixue/Documents/Master thesis/Statafile/Thesis2026/out/Tables/housing_tables/ethnicity_housing_shares.tex

\begin{table}[htbp]
\centering
\caption{Housing Conditions at Baseline by Ethnicity}
\label{tab:ethnicity_housing}
\begin{threeparttable}
\footnotesize
\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}p{5.4cm}ccccccc}
\toprule
& \multicolumn{3}{c}{Bedroom-per-person ratio} & \multicolumn{3}{c}{Number of other rooms} & \\
\cmidrule(lr){2-4}\cmidrule(lr){5-7}
Ethnicity & <1 & 1--<2 & $\geq$2 & 0--1 & 2--3 & 4+ & N \\
\midrule
British/English/Scottish/Welsh/Northern Irish & 22.91 & 54.70 & 22.38 & 39.52 & 52.25 & 8.23 & 12,385 \\
Indian & 46.87 & 45.80 & 7.33 & 37.76 & 56.40 & 5.84 & 417 \\
Pakistani & 76.40 & 20.30 & 3.30 & 48.19 & 48.45 & 3.36 & 250 \\
Bangladeshi & 78.71 & 20.64 & 0.65 & 74.50 & 23.37 & 2.13 & 92 \\
African & 35.30 & 61.13 & 3.56 & 63.06 & 35.48 & 1.45 & 107 \\
Caribbean & 19.62 & 58.71 & 21.66 & 65.77 & 31.33 & 2.90 & 129 \\
Any 